# 課程 03 - 代理設計模式

在本課程中，我們將探索三個建立有效人工智能代理的基礎設計模式：

1. <strong>明確的代理指令</strong> — 編寫精確、定義角色的提示，指導代理行為
2. **使用 Pydantic 模型的結構化輸出** — 確保代理返回可預測且驗證過的數據
3. <strong>單一職責代理</strong> — 設計專注且擅長單一任務的代理

我們將把每個模式應用於<strong>旅遊目的地推薦</strong>的場景，逐步構建一個能建議目的地、檢查可用性及處理後勤的系統。


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 模式 1：清晰的代理人指示

最有影響力的模式也是最簡單的：為你的代理人撰寫清晰且詳細的指示。

良好的指示會定義：
- <strong>代理人是誰</strong>（人物設定和語氣）
- <strong>它應該做什麼</strong>（逐步的責任）
- <strong>它應該如何表現</strong>（限制和風格）

以下，我們建立一個旅行禮賓代理人，並用明確的指示來塑造它每一次回應的內容。


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## 範例 2：使用 Pydantic 模型的結構化輸出

自由格式文字對話很有用，但下游系統需要結構化數據。
通過將 **Pydantic 模型** 與 <strong>工具函數</strong> 配對，我們可以：

- 為代理的輸出定義精確的結構
- 自動驗證回應
- 可靠地將代理結果整合到應用邏輯中

強制執行的關鍵是執行代理時傳入 `response_format`。這會強迫
模型返回經驗證的 `TravelRecommendations` 物件（存在於 `response.value`）
而非自由格式文字。`get_destination_details` 工具也返回類型化的
`DestinationRecommendation`，因此數據端到端保持結構化。


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## 模式 3：單一職責代理

複雜任務收益於分拆工作給多個專注的代理，每個代理有單一職責：

- 一個了解地點和可用性的 <strong>目的地專家</strong>
- 一個處理航班、酒店和行程的 <strong>物流規劃師</strong>

這反映了軟件工程中的 <em>關注點分離</em> 原則——每個代理更容易獨立測試、維護和改進。


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## 摘要

在本課程中，我們將三種具代理性的設計模式應用於旅遊推薦場景：

| 模式 | 主要概念 | 優點 |
|---|---|---|
| <strong>清晰指示</strong> | 預先定義角色、職責及限制 | 穩定、一致且符合品牌的代理行為 |
| <strong>結構化輸出</strong> | 使用 Pydantic 模型作為回應格式 | 經過驗證、機器可讀的結果 |
| <strong>單一職責</strong> | 給每個代理明確聚焦的工作 | 更易測試、維護及組合 |

這些模式能自然組合——你可以在單一職責的代理中結合清晰指示與結構化輸出，構建穩健且可用於生產的系統。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
